In [1]:
import sys
sys.path.append("..")

In [49]:
import tqdm
import torch
import pickle
import kaleido
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
def runSparsity (params: dict, diff_const = 1e-2):
    first_time = True

    for algorithm in params['algorithms']:
        for v_alpha in params['alphas']:
            for v_lamb in params['lambdas']:
                for seed in params['seeds']:

                    file_path = f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl"
                    data_tmp = pd.read_pickle(file_path)
                    if params['include_mask'] and "L1PSD" in params["algorithms"] and algorithm != "L1PSD":
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                        mask = data_l1psd["i"].to_numpy()
                        data_tmp = data_tmp.iloc[mask]
                    else:
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_0.1_0.1_{seed}.pkl")
                        mask_i = data_l1psd["i"].to_numpy()
                        if params['data'] == "sba" and params['base_model'] == "lr" and v_alpha == 0.5:
                            data_tmp = data_tmp[data_tmp['i'].isin(mask_i)]    
                        elif params['data'] == "sba" and params['base_model'] == "nn" and v_alpha == 0.5:
                            data_tmp = data_tmp[data_tmp['i'].isin(mask_i)]

                    if first_time:
                        data = data_tmp.copy(deep=True)
                        first_time = False
                    else:
                        data = pd.concat((data, data_tmp), ignore_index=True)

                    print(f"Read {file_path}")

    data['add_diff'] = np.abs(data['x_r'] - data['x_0'])
    data['add_diff_count'] = data['add_diff'].apply(lambda x: np.sum(x > diff_const))
    data['multi_diff'] = np.abs(data['x_r'] - data['x_0']) / (np.abs(data['x_0']) + 1e-7)
    data['multi_diff_count'] = data['multi_diff'].apply(lambda x: np.sum(x > 0.01))

    return data

In [208]:
params = {}
# 'lr', 'nn'
params['base_model'] = 'nn'
# 'synthetic', 'german', 'sba'
params['data'] = 'german'
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
params['include_mask'] = True


params['alphas']= [0.1]
# german_lr (alpha=0.1)
# params['lambdas'] = [0.5,0.3,0.1,0.04,0.01,0.004,0.001]
# german_nn (alpha=0.1)
params['lambdas'] = [0.7, 0.3, 0.1, 0.05, 0.01, 0.001]
# sba_lr (alpha=0.1)
# params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
# sba_nn (alpha=0.1)
# params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]

# params['alphas']= [0.5]
# # german_lr (alpha=0.5)
# # params['lambdas'] = [0.3,0.04, 0.004, 0.1, 0.01]
# # german_nn (alpha=0.5)
# # params['lambdas'] =  [3.0, 0.7, 0.3, 0.1, 0.05, 0.01]
# # sba_lr (alpha=0.5)
# # params['lambdas'] = [0.01, 0.1, 0.7, 1.4, 2.1]
# # sba_nn (alpha=0.5)
# params['lambdas'] = [0.01, 0.1, 0.7, 2.1, 3.5]

sparsity_lambda = deepcopy(params['lambdas'])
sparsity_lambda.sort(reverse=True)
diff_const = 1e-2
df_results = runSparsity(params, diff_const=diff_const)

Read ../results/recourse/nn_german_Alg1_0.7_0.1_0.pkl
Read ../results/recourse/nn_german_Alg1_0.7_0.1_1.pkl
Read ../results/recourse/nn_german_Alg1_0.7_0.1_2.pkl
Read ../results/recourse/nn_german_Alg1_0.7_0.1_3.pkl
Read ../results/recourse/nn_german_Alg1_0.7_0.1_4.pkl
Read ../results/recourse/nn_german_Alg1_0.3_0.1_0.pkl
Read ../results/recourse/nn_german_Alg1_0.3_0.1_1.pkl
Read ../results/recourse/nn_german_Alg1_0.3_0.1_2.pkl
Read ../results/recourse/nn_german_Alg1_0.3_0.1_3.pkl
Read ../results/recourse/nn_german_Alg1_0.3_0.1_4.pkl
Read ../results/recourse/nn_german_Alg1_0.1_0.1_0.pkl
Read ../results/recourse/nn_german_Alg1_0.1_0.1_1.pkl
Read ../results/recourse/nn_german_Alg1_0.1_0.1_2.pkl
Read ../results/recourse/nn_german_Alg1_0.1_0.1_3.pkl
Read ../results/recourse/nn_german_Alg1_0.1_0.1_4.pkl
Read ../results/recourse/nn_german_Alg1_0.05_0.1_0.pkl
Read ../results/recourse/nn_german_Alg1_0.05_0.1_1.pkl
Read ../results/recourse/nn_german_Alg1_0.05_0.1_2.pkl
Read ../results/recourse/

In [209]:
df_results['algorithm'] = df_results['algorithm'].replace('alg1',"Alg1")
df_results['algorithm'] = df_results['algorithm'].replace('ROAR',"ROARLInf")

df_results_mean = df_results.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()

In [210]:
# df_results_mean = df_results_mean.sort_values(['alpha', 'lambda', 'algorithm'])
# df_results_mean_sam = df_results_mean.iloc[0:len(params['algorithms'])]
# if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
#     df_results_mean_sam.iloc[[2,3]] = df_results_mean_sam.iloc[[3,2]]
# stacked  = np.stack(df_results_mean_sam['diff'].apply(lambda x: np.where(x > diff_const, x, 0)))

# fig = px.imshow(stacked.round(1), 
#             color_continuous_scale="Reds",
#             y=df_results_mean_sam['algorithm'].to_list(),
#             text_auto=True,
#             labels=dict(x='Features', y='Algorithms', color='Avg Cost'),
#             title=f"German LR Alpha={df_results_mean_sam['alpha'].unique()} Lambda={df_results_mean_sam['lambda'].unique()}")

# fig.show()

In [213]:
df_results_mean_histo = df_results_mean.copy(deep=True)
df_results_mean_histo['lambda_str'] = df_results_mean_histo['lambda'].astype(str)

df_results_mean = df_results_mean.sort_values(['lambda', 'algorithm'], ascending=[False, False])
df_results_mean

,algorithm,alpha,lambda,seed,i,x_0,x_r,theta_0,add_diff,add_diff_count,multi_diff,multi_diff_count
23,ROARLInf,0.1,0.700,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.8454579247368706, 0.7074209319220649, -0.30...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.06319097636309198, 0.06366805305226392, 0.0...",0.583333,"[0.07095931358684678, 0.2178780862968865, 0.01...",3.534722
17,ROARL1,0.1,0.700,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.8593173556857638, 0.7028757201300727, -0.30...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.049329861723962795, 0.05483681473914118, 0....",0.340278,"[0.04056758619547375, 0.214684583140642, 0.007...",3.368056
11,L1PSD,0.1,0.700,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.7363555555555558, 0.7499951388888888, -0.31...","[-1.209702777777778, 0.9730298611111111, 0.550...","[0.17225555555555555, 0.18436875000000003, 4.1...",0.520833,"[0.10284868404159798, 0.4464565915761386, 4.08...",0.520833
5,Alg1,0.1,0.700,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.6547298611111111, 0.7652263888888888, -0.30...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.25388125000000006, 0.2226347222222222, 0.00...",0.666667,"[0.16613630611790292, 0.41982575260262117, 0.0...",0.673611
22,ROARLInf,0.1,0.300,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.6299452251858182, 0.8372974395751953, -0.16...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.27866875254300216, 0.2858930522944648, 0.14...",1.687500,"[0.5261902601398758, 0.9141278423953926, 0.168...",4.291667
16,ROARL1,0.1,0.300,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.6820542017618815, 0.8296657138400607, -0.22...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.22658054914489123, 0.2466506936820379, 0.08...",1.493056,"[0.40186419681529073, 0.8404386118664879, 0.09...",4.243056
10,L1PSD,0.1,0.300,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.40260694444444456, 0.8902243055555554, -0.2...","[-1.209702777777778, 0.9730298611111111, 0.550...","[0.5060041666666666, 0.47620069444444435, 0.04...",0.895833,"[0.4666778115187244, 1.1136624365560772, 0.040...",0.895833
4,Alg1,0.1,0.300,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.18912777777777776, 0.8524729166666668, -0.2...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.7194833333333333, 0.5297312500000001, 0.015...",1.041667,"[0.6704660436100275, 0.8794515095075369, 0.013...",1.048611
21,ROARLInf,0.1,0.100,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.3163290818532308, 0.8974311616685655, 0.096...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.5922874982570515, 0.5780465249819683, 0.439...",2.930556,"[1.1559525698352742, 1.5419619912286384, 0.787...",5.402778
15,ROARL1,0.1,0.100,1.916667,14.576389,"[0.9086111111111114, 0.6608305555555556, -0.31...","[0.4202229181925456, 0.8958375718858507, 0.043...","[-1.195727777777777, 0.9590986111111115, 0.554...","[0.4883895850695876, 0.49978889452674324, 0.37...",2.812500,"[1.1286461899703883, 1.4174591716055205, 0.771...",5.243056


In [214]:
fig = px.histogram(df_results_mean_histo, 
                   x = "lambda_str",
                   y = "add_diff_count",
                   color="algorithm",
                   barmode="group",
                   title=f"{params['data']}_{params['base_model']}_alpha=0.1_histogram")
fig.show()

In [215]:
'#636EFA',
'#EF553B',
'#00CC96',
'#AB63FA',
"#8EF1F3",
"#F6B08C",
"#B5FFBE",
"#D7BBF4"

custom_colors = {"Alg1" : '#636EFA', "L1PSD" : '#EF553B',"ROARLInf" : '#00CC96', "ROARL1" : '#AB63FA'}
algorithm_latex_map = {"Alg1" : r"$\text{Alg}2\ (L^\infty)\ (\alpha=0.1)$",
                "L1PSD" : r"$\text{Alg}1\ (L^1)\ (\alpha=0.1)$",
                "ROARLInf" : r"$\text{ROAR}\ (L^\infty)\ (\alpha=0.1)$",
                "ROARL1" : r"$\text{ROAR}\ (L^1)\ (\alpha=0.1)$"}
params["algorithms"] = ["L1PSD", "Alg1", "ROARL1", "ROARLInf"]

In [216]:
fig = go.Figure()
font_family = 'Times New Roman'
font_color = 'black'
font_size = 20
# width, height = 720, 540
width, height = 720, 540

for algo in params['algorithms']:
    df_sub = df_results_mean_histo[df_results_mean_histo["algorithm"] == algo]
    fig.add_trace(
        go.Bar(
            x=df_sub["lambda_str"],
            y=df_sub["add_diff_count"],
            name=algorithm_latex_map[algo],
            marker=dict(color=custom_colors[algo])
        )
    )

fig.update_layout(plot_bgcolor="white",
                  paper_bgcolor="white",)
                #   xaxis=dict(title="Lambda",
                #              showgrid=True,
                #              mirror=True,
                #              gridcolor="black",),
                #     yaxis=dict(title="sum of diff_count",
                #                showgrid=True,
                #                gridcolor="black",
                #                mirror=True))
fig.update_xaxes(
        title=dict(
            text= r"$\lambda$",
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            )
            ), 
        showline=True, 
        mirror=True,
        showgrid=False,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_yaxes(
        title=dict(
            text='No. of Features Changed',
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        # title =dict(
        #     text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
        #     x= 0.5, 
        #     font=dict(family=font_family, size=20)
        #     ),
        legend=dict(
            # x=0.975,
            x=0.025, 
            y=0.975, 
            orientation='v',
            xanchor='left',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            range=[0,7]
        )
    )

In [217]:
figName = f"sparsity-addictive-model_{params['base_model']}-dataset_{params['data']}-alpha_0.1" 
kaleido.write_fig_sync(fig, f"../fig/{figName}.pdf")

In [219]:
fig = go.Figure()
font_family = 'Times New Roman'
font_color = 'black'
font_size = 20
# width, height = 720, 540
width, height = 720, 540

for algo in params['algorithms']:
    df_sub = df_results_mean_histo[df_results_mean_histo["algorithm"] == algo]
    fig.add_trace(
        go.Bar(
            x=df_sub["lambda_str"],
            y=df_sub["multi_diff_count"],
            name=algorithm_latex_map[algo],
            marker=dict(color=custom_colors[algo])
        )
    )

fig.update_layout(plot_bgcolor="white",
                  paper_bgcolor="white",)
                #   xaxis=dict(title="Lambda",
                #              showgrid=True,
                #              mirror=True,
                #              gridcolor="black",),
                #     yaxis=dict(title="sum of diff_count",
                #                showgrid=True,
                #                gridcolor="black",
                #                mirror=True))
fig.update_xaxes(
        title=dict(
            text=r"$\lambda$",
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            )
            ), 
        showline=True, 
        mirror=True,
        showgrid=False,
        linecolor='black', 
        gridcolor='lightgrey', 
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_yaxes(
        title=dict(
            text='No. of Features Changed',
            font=dict(
                family=font_family,
                color=font_color,
                size=font_size
            ), 
            ), 
        showline=True, 
        mirror=True,
        linecolor='black', 
        gridcolor='lightgrey',
        zerolinewidth=1,
        zerolinecolor='lightgrey',
        )

fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(t=50,b=25,l=25,r=25),
        # title =dict(
        #     text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC ({params['adv_method']}) Adversary", 
        #     x= 0.5, 
        #     font=dict(family=font_family, size=20)
        #     ),
        legend=dict(
            # x=0.975,
            x=0.025, 
            y=0.975, 
            orientation='v',
            xanchor='left',
            font=dict(
                family=font_family,
                color=font_color,
                size=15
                ), 
            bgcolor='rgba(255, 255, 255, 0.7)',
            bordercolor='lightgrey',
            borderwidth=1,
            entrywidth=100.5,
            ),
        xaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20,
            ),
            
        ),
        yaxis=dict(
            tickfont=dict(
                family=font_family,
                color=font_color,
                size=20
            ),
            range=[0,7]
        )
    )

In [ ]:
figName = f"sparsity-multiplicative-model_{params['base_model']}-dataset_{params['data']}-alpha_0.1" 
kaleido.write_fig_sync(fig, f"../fig/{figName}.pdf")

In [105]:
# figName = f"sparsity-model_{params['base_model']}-dataset_{params['data']}-alpha_0.1" 
# fig.write_image(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
# f"experimentResults\\figures\\" + 
# figName + f".pdf")

In [285]:
from plotly.subplots import make_subplots

width_oneF = 150
height_oneF = 1000 / len(params["algorithms"])

fig_sub = make_subplots(rows = len(sparsity_lambda), 
                        cols = 1, 
                        shared_xaxes=True,
                        subplot_titles=[f"Lambda {val}" for val in sparsity_lambda],
                        x_title="Features",
                        y_title="Algorithms")

for i,lamb in enumerate(sparsity_lambda):
    df_results_mean_tmp = df_results_mean[df_results_mean['lambda'] == lamb]
    if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
        df_results_mean_tmp.iloc[[0,1]] = df_results_mean_tmp.iloc[[1,0]]
    stacked  = np.stack(df_results_mean_tmp['diff'])

    fig_sub.add_trace(go.Heatmap(z=stacked.round(2),
                                x=np.arange(stacked.shape[1]),
                                y=df_results_mean_tmp['algorithm'].to_list(),
                                coloraxis="coloraxis",
                                texttemplate="%{z}"), 
                                row=i+1, col=1)

fig_sub.update_xaxes(tickmode="array", 
                     tickvals=np.arange(stacked.shape[1]), 
                     row=len(sparsity_lambda), 
                     col=1)
# fig_sub.update_yaxes(title_text="Algorithms", row=len(sparsity_lambda) // 2, col=1)
fig_sub.update_layout(
    coloraxis=dict(colorscale="Reds"),
    coloraxis_colorbar=dict(
        title="Avg. Cost",
    ),
    width=width_oneF * stacked.shape[1],
    height=height_oneF * stacked.shape[0],
    title_text = f"{params['data']}_{params['base_model']}_alpha=0.1_Sparsity"
)

In [ ]:
# figNameHisto = f"{params['base_model']}_{params['data']}_histogram.html" 
# figNameSparsity = f"{params['base_model']}_{params['data']}_sparsity.html"
# fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameHisto + f".html")
# fig_sub.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameSparsity + f".html")

In [278]:
df_results[(df_results['seed'] == 0) & (df_results['i'] == 24) & (df_results['lambda'] == 0.001)]

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0,diff,diff_count
175,Alg1,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[1.2526, 5.1217, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-1.3217, 1.6803, 0.7039, -0.1972, -0.0802, 0....","[0.0, 5.4578, 0.0, 0.0, 0.0, 0.0, 0.0]",1
385,L1PSD,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-0.728, 2.7053, -1.0155, 0.0, 0.0, -0.0, 1.0]","[-1.4307, 1.8215, 0.7708, -0.1706, -0.1865, 0....","[1.9806, 3.0414, 0.0, 0.0, 0.0, 0.0, 0.0]",2
